# Data preparation

## Running the Propp pipeline for every book in La comédie humaine

In [ ]:
def process_book(book_path, spacy_model, mentions_detection_model, coreference_resolution_model):

    ######################################################################################

    from propp_fr import load_text_file
    text_content = load_text_file(book_path)

    ######################################################################################

    from propp_fr import generate_tokens_df
    tokens_df = generate_tokens_df(text_content, spacy_model)

    ######################################################################################

    from propp_fr import load_tokenizer_and_embedding_model, get_embedding_tensor_from_tokens_df

    # Load the tokenizer and pre-trained embedding model
    tokenizer, embedding_model = load_tokenizer_and_embedding_model(
        mentions_detection_model["base_model_name"],
      )

    # Generate embeddings for all tokens
    tokens_embedding_tensor = get_embedding_tensor_from_tokens_df(
        text_content,
        tokens_df,
        tokenizer,
        embedding_model,
      )

    ######################################################################################

    from propp_fr import generate_entities_df

    entities_df = generate_entities_df(
        tokens_df,
        tokens_embedding_tensor,
        mentions_detection_model,
    )

    ######################################################################################

    from propp_fr import add_features_to_entities

    entities_df = add_features_to_entities(entities_df, tokens_df)

    ######################################################################################

    from propp_fr import perform_coreference

    entities_df = perform_coreference(
        entities_df,
        tokens_embedding_tensor,
        coreference_resolution_model,
        )

    ######################################################################################

    from propp_fr import extract_attributes
    tokens_df = extract_attributes(entities_df, tokens_df)

    ######################################################################################

    from propp_fr import generate_characters_dict

    characters_dict = generate_characters_dict(tokens_df, entities_df)

    ######################################################################################

    return tokens_df, entities_df, characters_dict

In [ ]:
import os
import gc
import datetime
import traceback
import torch
from propp_fr import load_models, save_tokens_df, save_entities_df, save_book_file

# Загружаем модели один раз для всех книг
spacy_model, mentions_detection_model, coreference_resolution_model = load_models()

for subdir, dirs, files in os.walk("la_comedie_humaine"):
    for file in files:
        if not file.endswith(".txt"):
            continue

        book_name = file[:-4]
        book_path = os.path.join(subdir, file)

        # Пропускаем уже обработанные файлы
        book_file_path = os.path.join(subdir, book_name + "_book_file.book")
        if os.path.exists(book_file_path):
            print(f"Skipping (already processed): {file}")
            continue

        print("####################################################################################")
        print(f"Processing file: {file}")
        print(f"Start time: {datetime.datetime.now()}")

        try:
            tokens, entities, characters = process_book(
                book_path, spacy_model, mentions_detection_model, coreference_resolution_model
            )
            save_tokens_df(tokens, book_name + "_tokens", subdir)
            save_entities_df(entities, book_name + "_entities", subdir)
            save_book_file(characters, book_name + "_book_file", subdir)
            print(f"End time: {datetime.datetime.now()}")
        except Exception as e:
            print(f"ERROR processing {file}: {e}")
            traceback.print_exc()
        finally:
            # Очищаем память после каждой книги (и после ошибок тоже)
            gc.collect()
            torch.cuda.empty_cache()